# Milestone 3 : moteur de consolidation

Ce notebook construit le moteur de consolidation des livraisons : à partir du planning des besoins des usines, il regroupe les lignes dans des camions, choisit pour chaque couple fournisseur × usine ses jours fixes de livraison, lisse la charge des usines sur la semaine et chiffre les économies réalisées par rapport à la situation actuelle (baseline du milestone 2).

### Définition des variables et des fonctions

In [ ]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Fichiers ----------
SILVER_CLEAN_PATH = "../data/silver/released_schedule_cleaned.csv"
QUARANTAINE_PATH = "../data/silver/released_schedule_quarantine.csv"
BASELINE_PATH = "../data/baseline/baseline_camion.csv"   # baseline du milestone 2 : une ligne par camion
SORTIE = Path("../data/gold")

# ---------- Paramètres du moteur ----------
POIDS_MAX = 24500                 # kg par camion
VOLUME_MAX = 110.4                # m³ par camion
FENETRE = pd.Timedelta(days=7)    # avance maximale d'une livraison sur le besoin
JOURS_FIXES_MAX = 2               # un couple livre 1 ou 2 jours fixes par semaine
SEUIL_LISSAGE = 0.35              # écart max toléré entre un jour et la moyenne de la semaine
MIN_CAMIONS_SEMAINE = 5           # semaines plus petites (moins d'un camion par jour) : pas contrôlées
JOURS = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi"]

# ---------- Coût de transport ----------
FREIGHT_VALUE_MARGIN = 2.3923     # +239,23 %
DIESEL_SURCHARGE = 1.1943
couts = pd.read_csv("../data/bronze/rate_sheet.csv", index_col="km_max")
TRANCHES_KM = [int(km) for km in couts.index if not km.startswith(">")]    # 30, 50, ..., 650
TRANCHES_KG = [int(kg) for kg in couts.columns if not kg.startswith(">")]  # 50, 100, ..., 4000
distances = pd.read_csv("../data/bronze/distances.csv", encoding="utf-8-sig").set_index("supplier_id")["distance"]

# Données manquantes : estimations hautes (comme dans la baseline du milestone 2)
DISTANCE_PAR_DEFAUT = TRANCHES_KM[-1] + 1  # 651 km -> tranche "> 650 km"
POIDS_PAR_DEFAUT = TRANCHES_KG[-1] + 1     # 4 001 kg -> tranche "> 4 000 kg"
SANS_DISTANCE = sorted(distances.index[distances.isna()])
distances = distances.fillna(DISTANCE_PAR_DEFAUT)


def tranche(valeur, tranches):
    # Première tranche >= valeur (on majore), sinon la tranche '>max'
    for borne in tranches:
        if valeur <= borne:
            return str(borne)
    return f">{tranches[-1]}"


def cout_transport(poids, distance):
    # Prix d'un camion en € = coût de la grille × (1 + marge) × diesel
    cout_base = couts.loc[tranche(distance, TRANCHES_KM), tranche(poids, TRANCHES_KG)]
    return float(round(cout_base * (1 + FREIGHT_VALUE_MARGIN) * DIESEL_SURCHARGE, 2))


# ---------- Mise en forme ----------
def nombre(x):
    return f"{x:,.1f}".replace(",", " ").replace(".", ",").removesuffix(",0")


def pourcent(x):
    return f"{x:.1%}".replace(".", ",").replace("%", " %")


def eur(x):
    return f"{x:,.0f} €".replace(",", " ")

## 1. Contraintes et objectif

### Contraintes

Les contraintes dures ne sont jamais violées. La contrainte de lissage est souple : le moteur la poursuit, mais une semaine peut rester hors tolérance quand rien ne permet de l'équilibrer (camions de démarrage, voir partie 4).

| # | Contrainte | Type | Fonction |
|---|---|---|---|
| C1 | Un camion ne dépasse pas 24,5 t | dure | `c1_poids` |
| C2 | Un camion ne dépasse pas 110,4 m³ | dure | `c2_volume` |
| C3 | Chaque ligne est livrée, une seule fois, entière (lots indivisibles) | dure | `c3_exhaustivite` |
| C4 | Un camion ne transporte qu'un seul couple | dure | `c4_un_couple` |
| C5 | Jamais de retard | dure | `c5_retard` |
| C6 | Au plus 7 jours d'avance | dure | `c6_avance` |
| C7 | Livraison un jour ouvré, dans l'horizon du planning | dure | `c7_jours_ouvres` |
| C8 | Jours fixes : 1 ou 2 jours par couple, et le camion part un de ces jours | dure | `c8_jours_fixes` |
| C9 | Au plus 1 camion par couple et par jour, sauf si le besoin d'un seul jour dépasse 24,5 t | dure | `c9_un_camion_par_jour` |
| C10 | Lissage : dans chaque usine, aucun jour ne s'écarte de plus de 35 % (et d'au moins 1 camion) de la moyenne de la semaine ; semaines d'au moins 5 camions | souple | `c10_lissage` |

Deux cas particuliers, fixés par les hypothèses des milestones 1 et 2 :
- camions de démarrage : les besoins du début d'horizon qui tombent avant le premier jour fixe du couple ne peuvent pas être livrés avant le 09/11. Ils partent le jour du besoin et sont exemptés de C8 ;
- lignes de la quarantaine sans poids : elles ne peuvent pas être consolidées. Chacune a son camion réservé, le jour du besoin, chiffré en tranche > 4 000 kg. Ces camions sont exemptés de C1 (poids inconnu), C8 et C9, mais ils comptent dans la charge des usines (C10).

### Objectif

Minimiser le coût total de transport :

$$\min \sum_{c} \text{coût}(c), \qquad \text{coût}(c) = \text{grille}\big[\text{tranche}(\text{distance}_{k(c)}),\ \text{tranche}(\textstyle\sum_{l \in c} w_l)\big] \times (1 + 2{,}3923) \times 1{,}1943$$

La grille donne un prix par tranche de distance et de poids : un camion plus plein coûte un peu plus cher, mais beaucoup moins que deux camions. Regrouper les lignes dans moins de camions est donc ce qui fait baisser le coût.

### Fonction de vérification des contraintes

In [ ]:
# ---------- Fonction objectif ----------
def cout_camion(poids, supplier_id):
    # Coût d'un camion ; poids inconnu (camion réservé) -> tranche > 4 000 kg
    return cout_transport(POIDS_PAR_DEFAUT if pd.isna(poids) else poids, distances[supplier_id])


def cout_total(planning):
    # Objectif à minimiser : somme des coûts des camions
    return sum(cout_camion(p, s) for p, s in zip(planning["poids"], planning["supplier_id"]))


# ---------- Charge des usines (utilisée par C10 et par le lissage) ----------
def case(date):
    # (n° de semaine dans l'horizon, jour de la semaine) d'une date
    return SEMAINES.index(date.isocalendar().week), date.day_of_week


def charge_depuis_camions(camions, plant):
    # Camions reçus par une usine : une ligne par semaine, une colonne par jour (lundi -> vendredi)
    charge = np.zeros((NB_SEMAINES, 5))
    for d in camions.loc[camions["plant"] == plant, "date"]:
        if d.isocalendar().week in SEMAINES and d.day_of_week < 5:
            charge[case(d)] += 1
    return charge


def charges_camions(camions):
    return {plant: charge_depuis_camions(camions, plant) for plant in USINES}


def ecart(charge_jours):
    # Écart du jour le plus éloigné de la moyenne, en % de la moyenne (0.35 = 35 %)
    charge_jours = np.asarray(charge_jours, dtype=float)
    moyenne = charge_jours.mean()
    return 0.0 if moyenne == 0 else float(np.abs(charge_jours - moyenne).max() / moyenne)


def hors_tolerance(charge_semaines, seuil=None):
    # C10 pour chaque semaine : un jour s'écarte-t-il de la moyenne de plus du seuil (et d'au moins 1 camion) ?
    seuil = SEUIL_LISSAGE if seuil is None else seuil
    moyenne = charge_semaines.mean(axis=1, keepdims=True)
    tolerance = np.maximum(seuil * moyenne, 1.0)
    depasse = (np.abs(charge_semaines - moyenne) > tolerance + 1e-9).any(axis=1)
    return depasse & (charge_semaines.sum(axis=1) >= MIN_CAMIONS_SEMAINE)


# ---------- Vérification des contraintes ----------
# Chaque fonction retourne les cas qui violent la contrainte (table vide = contrainte respectée).
# planning : une ligne par camion (colonnes plant, supplier_id, date, poids, volume, demarrage, reserve, profil)
# lignes   : une ligne par ligne de besoin (colonnes id_ligne, camion_id, plant, supplier_id, need_date,
#            date_livraison, weight)

def optimises(planning):
    # Camions construits par le moteur (sans les camions réservés de la quarantaine)
    return planning[~planning["reserve"]]


def c1_poids(planning, lignes):
    p = optimises(planning)
    return p[p["poids"] > POIDS_MAX + 1e-6]


def c2_volume(planning, lignes):
    return planning[planning["volume"] > VOLUME_MAX + 1e-6]


def c3_exhaustivite(planning, lignes, ids_attendus):
    # Lignes sans camion, livrées deux fois, ou absentes
    sans_camion = lignes[lignes["camion_id"].isna()].assign(probleme="sans camion")
    doublons = lignes[lignes["id_ligne"].duplicated(keep=False)].assign(probleme="livrée plusieurs fois")
    absentes = pd.DataFrame({"id_ligne": sorted(set(ids_attendus) - set(lignes["id_ligne"])), "probleme": "absente"})
    inconnues = lignes[~lignes["id_ligne"].isin(ids_attendus)].assign(probleme="ligne inconnue")
    return pd.concat([sans_camion, doublons, absentes, inconnues], ignore_index=True)


def c4_un_couple(planning, lignes):
    nb_couples = lignes.groupby("camion_id")[["plant", "supplier_id"]].nunique().max(axis=1)
    return nb_couples[nb_couples > 1].rename("couples").reset_index()


def c5_retard(planning, lignes):
    return lignes[lignes["date_livraison"] > lignes["need_date"]]


def c6_avance(planning, lignes):
    return lignes[lignes["need_date"] - lignes["date_livraison"] > FENETRE]


def c7_jours_ouvres(planning, lignes):
    d = planning["date"]
    return planning[(d.dt.day_of_week >= 5) | (d < DEBUT) | (d > FIN)]


def c8_jours_fixes(planning, lignes):
    p = optimises(planning)
    mauvais_profil = ~p["profil"].map(len).between(1, JOURS_FIXES_MAX)
    hors_profil = np.array([not dem and d.day_of_week not in prof for d, prof, dem in zip(p["date"], p["profil"], p["demarrage"])], dtype=bool)
    return p[mauvais_profil | hors_profil]


def c9_un_camion_par_jour(planning, lignes):
    # Jours où un couple a 2 camions ou plus. 'justifie' = le besoin d'un seul jour du couple dépasse
    # 24,5 t : 2 camions sont alors inévitables
    par_jour = optimises(planning).groupby(["plant", "supplier_id", "date"]).size()
    trop = par_jour[par_jour > 1].rename("camions").reset_index()
    besoin_jour = lignes.dropna(subset=["weight"]).groupby(["plant", "supplier_id", "need_date"])["weight"].sum()
    impossibles = set(besoin_jour[besoin_jour > POIDS_MAX].index.droplevel("need_date"))
    trop["justifie"] = [(p, s) in impossibles for p, s in zip(trop["plant"], trop["supplier_id"])]
    return trop


def c10_lissage(planning, lignes):
    # Semaines hors tolérance, par usine (tous les camions reçus, démarrage et réservés compris)
    lignes_hors = []
    for plant, charge in charges_camions(planning).items():
        hors = hors_tolerance(charge)
        for w in CONTROLE:
            if hors[w]:
                lignes_hors.append({"plant": plant, "semaine": SEMAINES[w], "ecart": ecart(charge[w]),
                                    "camions_par_jour": charge[w].astype(int).tolist()})
    return pd.DataFrame(lignes_hors)


CONTRAINTES = [
    ("C1", "Poids ≤ 24,5 t", "dure", c1_poids),
    ("C2", "Volume ≤ 110,4 m³", "dure", c2_volume),
    ("C3", "Chaque ligne livrée une fois, entière", "dure", c3_exhaustivite),
    ("C4", "Un seul couple par camion", "dure", c4_un_couple),
    ("C5", "Jamais de retard", "dure", c5_retard),
    ("C6", "Au plus 7 jours d'avance", "dure", c6_avance),
    ("C7", "Jour ouvré, dans l'horizon", "dure", c7_jours_ouvres),
    ("C8", "Jours fixes (1 ou 2 par couple)", "dure", c8_jours_fixes),
    ("C9", "Au plus 1 camion par couple et par jour", "dure", c9_un_camion_par_jour),
    ("C10", "Lissage (écart ≤ 35 %)", "souple", c10_lissage),
]


def verifier_contraintes(planning, lignes, ids_attendus):
    # Tableau de synthèse : nombre de violations de chaque contrainte
    resultats = []
    for code_c, nom, type_c, fonction in CONTRAINTES:
        args = (planning, lignes, ids_attendus) if fonction is c3_exhaustivite else (planning, lignes)
        violations = fonction(*args)
        remarque = ""
        if fonction is c9_un_camion_par_jour:
            remarque = f"{int(violations['justifie'].sum())} cas inévitables (besoin d'un jour > 24,5 t)"
            violations = violations[~violations["justifie"]]
        if fonction is c10_lissage:
            remarque = f"sur {len(USINES) * len(CONTROLE)} semaines contrôlées"
        nb = len(violations)
        statut = "respectée" if nb == 0 else ("violée" if type_c == "dure" else f"{nb} semaine(s) hors tolérance")
        resultats.append({"Contrainte": code_c, "Description": nom, "Type": type_c, "Violations": nb,
                          "Statut": statut, "Remarque": remarque})
    return pd.DataFrame(resultats).set_index("Contrainte")

## 2. Choix de l'algorithme

Chaque membre du groupe a conçu son propre algorithme, sur les mêmes données et les mêmes contraintes. On les a comparés sur les mêmes critères, puis on a retenu celui qui donnait le meilleur résultat.

Deux approches ont été comparées au final :

| | Algorithme A : approche mécanique (règles) | Algorithme B : approche par couple et formulation mathématique |
|---|---|---|
| Principe | consolidation par couple avec une fenêtre de 7 jours ; chaque couple teste tous ses profils de jours fixes et garde le moins cher ; lissage par changements de profil au moindre surcoût ; report des lignes sur le jour fixe suivant | *à compléter* |
| Coût total (camions réservés compris) | 112 195 € | *à compléter* |
| Camions | 736 | *à compléter* |
| Économie vs baseline | 65 126 € (-36,7 %) | *à compléter* |
| Semaines hors tolérance | 4 / 20 | *à compléter* |
| Contraintes dures respectées | toutes | *à compléter* |
| Explicabilité | chaque décision est tracée (journal des changements de profil) | *à compléter* |
| Temps de calcul | quelques secondes | *à compléter* |

Algorithme retenu : *à compléter*.

*Justification à compléter quand le code de l'algorithme B sera disponible.*

La suite du notebook présente l'algorithme A.

## 3. Fonctionnement global

Le moteur prend en entrée le planning des besoins (une ligne = un matériau à livrer à une usine un jour donné) et produit le planning des camions. Il enchaîne quatre blocs :

```
Données préparées aux milestones 1 et 2 (besoins + lignes de la quarantaine)
        │
        ▼
Bloc 1 : jours fixes
  chaque couple teste ses 15 profils (5 à 1 jour, 10 à 2 jours), sans jamais 2 camions le même jour,
  et garde le moins cher ; le camion part le dernier jour fixe avant son premier besoin
  et prend les besoins des 7 jours suivants (24,5 t et 110,4 m³ max)
        │
        ▼
Bloc 2 : contrôle de l'équilibre
  pour chaque usine et chaque semaine : un jour s'écarte-t-il de plus de 35 % de la moyenne ?
        │ non ──► rien à faire
        │ oui
        ▼
Bloc 3 : lissage
  on change le profil (les jours fixes) de certains couples de l'usine, au moindre surcoût,
  jusqu'à repasser dans la tolérance
        │
        ▼
Bloc 4 : report et planning final
  les lignes qui peuvent attendre partent avec le camion du jour fixe suivant (s'il a la place
  et que ça ne coûte rien), puis on construit le planning complet
        │
        ▼
Validation des contraintes  ──►  Sortie 1 : profil de chaque couple
                                 Sortie 2 : contenu de chaque camion
```

Pourquoi cet ordre : on optimise d'abord le coût (bloc 1), car c'est l'objectif. Le lissage (bloc 3) ne fait ensuite que les changements nécessaires, là où une usine est déséquilibrée, en payant le moins possible. Le report (bloc 4) vient en dernier : il ne change ni les jours fixes ni le nombre de camions, il rapproche seulement des lignes de leur besoin.

| Paramètre | Valeur | Rôle |
|---|---|---|
| `POIDS_MAX`, `VOLUME_MAX` | 24 500 kg, 110,4 m³ | capacité d'un camion |
| `FENETRE` | 7 jours | avance maximale d'une livraison |
| `JOURS_FIXES_MAX` | 2 | nombre maximum de jours fixes par couple |
| `SEUIL_LISSAGE` | 35 % | écart toléré entre un jour et la moyenne de la semaine |
| `MIN_CAMIONS_SEMAINE` | 5 | en dessous, une semaine n'est pas contrôlée |

## 4. Moteur, bloc par bloc

On part des données préparées aux milestones 1 et 2 : les besoins nettoyés (silver), les lignes de la quarantaine et la baseline. La cellule ci-dessous les charge, crée un camion réservé pour chaque ligne de la quarantaine sans poids (comme dans la baseline du milestone 2), et fixe l'horizon du planning (semaines et usines).

In [ ]:
# ---------- Données des milestones 1 et 2 ----------
silver = pd.read_csv(SILVER_CLEAN_PATH, parse_dates=["need_date"])
quarantaine = pd.read_csv(QUARANTAINE_PATH, parse_dates=["need_date"])
N_SILVER = len(silver)
sans_poids = quarantaine["weight"].isna()
a_consolider = ~sans_poids & (quarantaine["weight"] <= POIDS_MAX)   # la ligne de plus de 24,5 t est exclue

# Besoins à consolider : silver + lignes de la quarantaine dont le poids est connu
besoins = (pd.concat([silver, quarantaine.loc[a_consolider, silver.columns]], ignore_index=True)
           .sort_values(["plant", "supplier_id", "need_date", "weight"], ascending=[True, True, True, False])
           [["need_date", "supplier_id", "material", "plant", "unloading_point", "schedule_quantity", "weight", "volume"]])

# Lignes sans poids : un camion réservé par ligne, le jour du besoin
lignes_reserves = quarantaine[sans_poids].drop(columns="flags").reset_index(drop=True)
lignes_reserves["numero"] = range(1, len(lignes_reserves) + 1)
lignes_reserves["id_ligne"] = "Q" + lignes_reserves["numero"].astype(str)
lignes_reserves["date_livraison"], lignes_reserves["avance_jours"] = lignes_reserves["need_date"], 0
camions_reserves = pd.DataFrame({
    "numero": lignes_reserves["numero"], "plant": lignes_reserves["plant"], "supplier_id": lignes_reserves["supplier_id"],
    "date": lignes_reserves["need_date"], "demarrage": False, "nb_lignes": 1, "poids": np.nan,
    "volume": lignes_reserves["volume"], "cout": [cout_camion(np.nan, s) for s in lignes_reserves["supplier_id"]],
    "motif": "poids inconnu"})
COUT_RESERVE, NB_RESERVE = camions_reserves["cout"].sum(), len(camions_reserves)


def avec_reserves(camions):
    return pd.concat([camions, camions_reserves.drop(columns="numero")], ignore_index=True)


# Baseline du milestone 2 (sans ses camions réservés, comptés à part : les mêmes pour toutes les versions)
camions_base = pd.read_csv(BASELINE_PATH, parse_dates=["date"])
assert camions_base["reserve"].sum() == NB_RESERVE
camions_base = camions_base[~camions_base["reserve"]].copy()
camions_base["cout"] = [cout_camion(p, s) for p, s in zip(camions_base["poids"], camions_base["supplier_id"])]
COUT_BASELINE = camions_base["cout"].sum()

# Horizon du planning
DEBUT, FIN = besoins["need_date"].min(), besoins["need_date"].max()   # lundi 09/11 -> vendredi 18/12
SEMAINES = sorted(besoins["need_date"].dt.isocalendar().week.unique())
NB_SEMAINES = len(SEMAINES)
CONTROLE = list(range(NB_SEMAINES - 1))   # la dernière semaine est incomplète : pas contrôlée
USINES = sorted(besoins["plant"].unique())
IDS_ATTENDUS = {f"L{i}" for i in besoins.index} | set(lignes_reserves["id_ligne"])

print(f"Besoins à consolider : {len(besoins)} lignes, {besoins.groupby(['plant', 'supplier_id']).ngroups} couples, "
      f"{len(USINES)} usines, du {DEBUT:%d/%m} au {FIN:%d/%m}")
print(f"Camions réservés (lignes sans poids) : {NB_RESERVE}, {eur(COUT_RESERVE)}")
print(f"Baseline : {len(camions_base)} camions, {eur(COUT_BASELINE)} (+ les camions réservés)")

### Bloc 1 : jours fixes

Rôle : donner à chaque couple ses jours fixes de livraison (1 ou 2 jours par semaine), en regroupant ses lignes dans le moins de camions possible.

Principe : pour un profil donné, la consolidation suit la même boucle que l'algorithme de base, avec une seule différence. Quand un camion s'ouvre, il ne part pas le jour du plus ancien besoin restant, mais le dernier jour fixe du couple avant ce besoin (ou le jour même). Il prend ensuite les besoins jusqu'à sa date + 7 jours, du plus ancien au plus récent, tant qu'il reste de la place (24,5 t, 110,4 m³).

```
Couple au profil « mardi + vendredi »
besoin restant le plus ancien : jeudi 19/11     ──►  camion le mardi 17/11 (prend les besoins du 17/11 au 24/11)
besoin restant le plus ancien : mercredi 25/11  ──►  camion le mardi 24/11 (prend les besoins du 24/11 au 01/12)
besoin restant le plus ancien : samedi 05/12    ──►  camion le vendredi 04/12 ...
```

Choix du profil : chaque couple teste ses 15 profils (5 à 1 jour, 10 à 2 jours). On ne garde que ceux où il n'a jamais 2 camions le même jour (C9, fonction `autorises`), puis on prend le moins cher (à coût égal : le moins de camions, puis le moins de jours). Si aucun profil ne le permet, parce que le besoin d'un seul jour dépasse 24,5 t, le couple garde tous ses profils et prend le moins cher.

Démarrage : un besoin du début d'horizon qui tombe avant le premier jour fixe du couple ne peut pas être livré avant le 09/11. Son camion part le jour du besoin (camion de démarrage).

Deux variantes sont construites pour mesurer l'intérêt d'un deuxième jour : 1 jour fixe seulement, et 1 ou 2 jours fixes. Elles sont comparées à la référence sans jours fixes (l'algorithme de base du milestone 2), qui revient à autoriser les livraisons les 5 jours de la semaine.

In [ ]:
PROFILS = [(j,) for j in range(5)] + list(itertools.combinations(range(5), JOURS_FIXES_MAX))
nom_profil = lambda profil: " + ".join(JOURS[j] for j in profil)


def date_camion(besoin, profil):
    # Dernier jour du profil avant ou égal au besoin ; s'il tombe avant l'horizon -> date du besoin (démarrage)
    date = max(besoin - pd.Timedelta(days=(besoin.day_of_week - j) % 7) for j in profil)
    return (besoin, True) if date < DEBUT else (date, False)


def consolider_couple(lignes, profil):
    # Consolidation d'un couple avec des jours fixes (lignes triées par date, puis poids décroissant)
    restantes = list(lignes.itertuples())
    camions = []
    while restantes:
        date, demarrage = date_camion(restantes[0].need_date, profil)
        poids, volume, contenu, i = 0.0, 0.0, [], 0
        while i < len(restantes):
            ligne = restantes[i]
            if ligne.need_date > date + FENETRE:
                break                                   # hors fenêtre de 7 jours : camion fermé
            if poids + ligne.weight > POIDS_MAX or volume + ligne.volume > VOLUME_MAX:
                i += 1                                  # ne rentre pas : ligne suivante
                continue
            poids, volume = poids + ligne.weight, volume + ligne.volume
            contenu.append(ligne.Index)
            restantes.pop(i)
            i = 0
        camions.append({"date": date, "demarrage": demarrage, "poids": poids, "volume": volume, "lignes": contenu})
    return pd.DataFrame(camions)


# Toutes les options : chaque couple × chaque profil
scenarios = {}
for (plant, supplier), lignes_couple in besoins.groupby(["plant", "supplier_id"]):
    for profil in PROFILS:
        camions = consolider_couple(lignes_couple, profil)
        camions["cout"] = [cout_camion(p, supplier) for p in camions["poids"]]
        scenarios[(plant, supplier, profil)] = camions
# Référence : l'algorithme de base du milestone 2 (sans jours fixes), où le camion part le jour du plus ancien
# besoin, revient à livrer les 5 jours de la semaine
REFERENCE = (0, 1, 2, 3, 4)
camions_algo = pd.concat([consolider_couple(lignes_couple, REFERENCE).assign(plant=plant, supplier_id=supplier)
                          for (plant, supplier), lignes_couple in besoins.groupby(["plant", "supplier_id"])], ignore_index=True)
camions_algo["profil"] = [REFERENCE] * len(camions_algo)
camions_algo["cout"] = [cout_camion(p, s) for p, s in zip(camions_algo["poids"], camions_algo["supplier_id"])]
COUT_ALGO = camions_algo["cout"].sum()
print(f"Référence sans jours fixes : {len(camions_algo)} camions, {eur(COUT_ALGO)}")

cout_option = {cle: c["cout"].sum() for cle, c in scenarios.items()}
nb_camions_option = {cle: len(c) for cle, c in scenarios.items()}
charge_option = {(p, s, prof): charge_depuis_camions(c[~c["demarrage"]].assign(plant=p), p)   # hors démarrage
                 for (p, s, prof), c in scenarios.items()}
max_jour = {cle: int(c.groupby("date").size().max()) for cle, c in scenarios.items()}   # camions max un même jour


def autorises(plant, supplier, profils_variante):
    # Profils de la variante qui ne mettent jamais 2 camions le même jour à ce couple (C9).
    # Si aucun ne le permet (besoin d'un seul jour > 24,5 t) : tous les profils de la variante
    ok = [p for p in profils_variante if max_jour[(plant, supplier, p)] <= 1]
    return ok if ok else list(profils_variante)


def choix_moins_cher(profils_variante):
    # Profil le moins cher de chaque couple (à égalité : moins de camions, puis moins de jours)
    choix = {}
    for cle in sorted((k for k in scenarios if k[2] in autorises(k[0], k[1], profils_variante)),
                      key=lambda k: (k[0], k[1], round(cout_option[k], 2), nb_camions_option[k], len(k[2]), k[2])):
        choix.setdefault(cle[:2], cle[2])
    return choix


cout_choix = lambda choix: sum(cout_option[(p, s, prof)] for (p, s), prof in choix.items())
VARIANTES = {"1 jour fixe": [p for p in PROFILS if len(p) == 1], "1 ou 2 jours fixes": PROFILS}
choix_initiaux = {nom: choix_moins_cher(profils) for nom, profils in VARIANTES.items()}

impossibles = sorted({k[:2] for k in scenarios} - {k[:2] for k in scenarios if max_jour[k] <= 1})
print(f"{len(scenarios)} options évaluées ({len(choix_initiaux['1 ou 2 jours fixes'])} couples × {len(PROFILS)} profils)")
print(f"Couples qui ne peuvent pas avoir 1 seul camion par jour (besoin d'un jour > 24,5 t) : {impossibles}")
for nom, choix in choix_initiaux.items():
    nb = sum(nb_camions_option[(p, s, prof)] for (p, s), prof in choix.items())
    dem = sum(int(scenarios[(p, s, prof)]["demarrage"].sum()) for (p, s), prof in choix.items())
    print(f"{nom:<20}: {nb} camions (dont {dem} de démarrage), {eur(cout_choix(choix))} "
          f"(+{eur(cout_choix(choix) - COUT_ALGO)} vs algorithme de base), "
          f"couples à 2 jours : {sum(len(prof) == 2 for prof in choix.values())}")

Résultat (hors camions réservés, identiques dans toutes les versions) :

| Variante | Camions | dont démarrage | Coût | Surcoût vs algorithme de base | Couples à 2 jours |
|---|---|---|---|---|---|
| 1 jour fixe | 611 | 63 | 79 859 € | +3 852 € | 0 |
| 1 ou 2 jours fixes | 573 | 52 | 77 242 € | +1 235 € | 61 |

Imposer des jours fixes coûte un peu par rapport à l'algorithme de base, où chaque camion part le jour exact du premier besoin. Autoriser un deuxième jour divise ce surcoût par 3 : le camion part plus près du premier besoin, donc sa fenêtre de 7 jours couvre plus de besoins à venir, et il faut moins de camions de démarrage.

Seuls 2 couples ne peuvent pas avoir un seul camion par jour : SUP_00037 → PLT_002 et SUP_00046 → PLT_004, dont le besoin d'un même jour dépasse 24,5 t.

### Bloc 2 : contrôle de l'équilibre

Rôle : repérer les usines qui reçoivent trop de camions certains jours et pas assez d'autres.

Principe : pour chaque usine et chaque semaine, on compte les camions reçus chaque jour (C10). Une semaine est hors tolérance si un jour s'écarte de la moyenne de la semaine de plus de 35 %, avec au moins 1 camion de marge : une usine qui reçoit 5 camions dans la semaine ne peut pas faire mieux qu'un par jour. Les semaines de moins de 5 camions ne sont pas contrôlées, et la dernière semaine (S51) non plus, car elle est incomplète.Les camions de démarrage sont laissés de côté, car ils ne peuvent pas être déplacés. Les camions réservés de la quarantaine sont comptés, car ils arrivent aux quais.

In [ ]:
CHARGE_RESERVEE = charges_camions(camions_reserves)   # charge fixe, que le lissage ne déplace pas


def charges_choix(choix, fond=None):
    # Charge de chaque usine pour un choix de profils (hors démarrage), avec les camions réservés
    fond = CHARGE_RESERVEE if fond is None else fond
    return {plant: fond[plant] + sum(charge_option[(p, s, prof)] for (p, s), prof in choix.items() if p == plant)
            for plant in USINES}


def tableau_controle(charges):
    # Par usine : camions moyens par jour, pire semaine et décision de lissage
    lignes = []
    for plant, charge in charges.items():
        hors = hors_tolerance(charge[CONTROLE])
        lignes.append({"plant": plant, **dict(zip(JOURS, charge.mean(axis=0))),
                       "Pire semaine": max(ecart(charge[w]) for w in CONTROLE),
                       "Semaines hors tolérance": int(hors.sum()), "Lissage nécessaire": bool(hors.any())})
    return pd.DataFrame(lignes).set_index("plant")


def heatmap(charges_par_etape, titre):
    # Camions reçus par semaine × jour pour chaque usine ; une ligne de graphiques par étape
    fig, axes = plt.subplots(len(charges_par_etape), len(USINES), figsize=(16, 3.8 * len(charges_par_etape)),
                             sharex=True, sharey=True, squeeze=False)
    for ligne, (etape, charges) in enumerate(charges_par_etape.items()):
        for col, plant in enumerate(USINES):
            ax, charge = axes[ligne, col], charges[plant]
            hors = hors_tolerance(charge)
            im = ax.imshow(charge, cmap="YlGnBu", vmin=0, vmax=20, aspect="auto")
            for w in range(NB_SEMAINES):
                for j in range(5):
                    v = int(charge[w, j])
                    ax.text(j, w, v, ha="center", va="center", color="white" if v > 12 else "black",
                            fontweight="bold" if hors[w] and w in CONTROLE else "normal")
            ax.set_xticks(range(5), [j[:3] for j in JOURS])
            ax.set_yticks(range(NB_SEMAINES), [f"S{s}" for s in SEMAINES])
            ax.set_title(f"{plant} {etape}\n{int(hors[CONTROLE].sum())} semaine(s) hors tolérance", fontsize=10)
    fig.colorbar(im, ax=axes, label="Camions reçus", shrink=0.8)
    fig.suptitle(f"{titre}\n(en gras : semaine hors tolérance, S{SEMAINES[-1]} non contrôlée)", fontsize=13, y=1.02)
    plt.show()


format_controle = {**{j: "{:.1f}" for j in JOURS}, "Pire semaine": pourcent}
for nom, choix in choix_initiaux.items():
    print(f"{nom:<20}: {tableau_controle(charges_choix(choix))['Semaines hors tolérance'].sum()} semaines hors tolérance "
          f"sur {len(USINES) * len(CONTROLE)}")
tableau_controle(charges_choix(choix_initiaux["1 ou 2 jours fixes"])).style.format(format_controle)

Résultat : avant lissage, les 20 semaines contrôlées sont hors tolérance, dans les deux variantes. Chaque couple a pris son jour le moins cher, sans regarder les autres couples de l'usine : beaucoup tombent sur les mêmes jours. Les 4 usines doivent être lissées.

### Bloc 3 : lissage

Rôle : ramener chaque semaine dans la tolérance, au moindre surcoût.

Principe : on ne déplace pas des camions un par un, on change les jours fixes d'un couple (son profil), pour que ses livraisons restent régulières. Pour chaque usine hors tolérance, on répète :
1. tester tous les changements de profil possibles d'un couple de l'usine (autre jour, autre paire de jours), parmi les profils autorisés (jamais 2 camions le même jour) ;
2. garder ceux qui rendent les semaines plus régulières, c'est-à-dire qui font baisser la dispersion : la somme, sur les semaines contrôlées, des carrés des écarts à la moyenne de la semaine ;
3. appliquer celui qui coûte le moins par unité d'amélioration (les changements gratuits d'abord) ;

jusqu'à ce qu'aucune semaine ne soit hors tolérance, ou qu'aucun changement n'améliore plus rien. Les usines déjà équilibrées ne sont pas touchées.


In [ ]:
def dispersion(charge):
    # Somme, sur les semaines contrôlées, des carrés des écarts à la moyenne de la semaine
    charge = charge[CONTROLE]
    return float(((charge - charge.mean(axis=1, keepdims=True)) ** 2).sum())


def lisser_par_couple(choix, plant, profils_variante, seuil=None, fond=None):
    # Change le profil de couples de `plant` jusqu'à ce qu'aucune semaine contrôlée ne soit hors tolérance
    choix = dict(choix)
    charge = charges_choix(choix, fond)[plant]
    journal = []
    while hors_tolerance(charge[CONTROLE], seuil).any():
        dispersion_actuelle, meilleur = dispersion(charge), None
        for (p, s), prof in choix.items():
            if p != plant:
                continue
            for prof2 in autorises(p, s, profils_variante):     # jamais 2 camions le même jour
                if prof2 == prof:
                    continue
                nouvelle = charge - charge_option[(p, s, prof)] + charge_option[(p, s, prof2)]
                gain = dispersion_actuelle - dispersion(nouvelle)
                if gain <= 1e-9:
                    continue
                surcout = cout_option[(p, s, prof2)] - cout_option[(p, s, prof)]
                cle = (max(surcout, 0) / gain, -gain)
                if meilleur is None or cle < meilleur["cle"]:
                    meilleur = {"cle": cle, "supplier_id": s, "de": prof, "vers": prof2, "surcout": surcout,
                                "charge": nouvelle}
        if meilleur is None:
            break                                               # plus aucun changement n'améliore les semaines
        choix[(plant, meilleur["supplier_id"])] = meilleur["vers"]
        charge = meilleur["charge"]
        journal.append({"plant": plant, "supplier_id": meilleur["supplier_id"], "de": nom_profil(meilleur["de"]),
                        "vers": nom_profil(meilleur["vers"]), "surcout": round(meilleur["surcout"], 2),
                        "semaines_hors_tolerance_apres": int(hors_tolerance(charge[CONTROLE], seuil).sum())})
    return choix, journal


def lisser(choix, profils_variante, seuil=None, fond=None):
    # Lisse toutes les usines qui ont au moins une semaine hors tolérance
    choix, journal = dict(choix), []
    for plant in USINES:
        if hors_tolerance(charges_choix(choix, fond)[plant][CONTROLE], seuil).any():
            choix, journal_u = lisser_par_couple(choix, plant, profils_variante, seuil, fond)
            journal += journal_u
    return choix, pd.DataFrame(journal)


choix_lisses, journaux = {}, {}
for nom, profils in VARIANTES.items():
    choix_lisses[nom], journaux[nom] = lisser(choix_initiaux[nom], profils)
    avant = tableau_controle(charges_choix(choix_initiaux[nom]))["Semaines hors tolérance"].sum()
    apres = tableau_controle(charges_choix(choix_lisses[nom]))["Semaines hors tolérance"].sum()
    print(f"{nom:<20}: {len(journaux[nom])} changements de profil, "
          f"surcoût {eur(cout_choix(choix_lisses[nom]) - cout_choix(choix_initiaux[nom]))} -> {eur(cout_choix(choix_lisses[nom]))} ; "
          f"semaines hors tolérance {avant} -> {apres} (sur {len(USINES) * len(CONTROLE)})")

PRINCIPALE = min(VARIANTES, key=lambda nom: cout_choix(choix_lisses[nom]))   # la moins chère après lissage
print(f"\nVariante retenue : {PRINCIPALE}")
tableau_controle(charges_choix(choix_lisses[PRINCIPALE])).style.format(format_controle)

In [ ]:
# Journal des changements de profil de la variante retenue (traçabilité)
journaux[PRINCIPALE]

In [ ]:
heatmap({"avant lissage": charges_choix(choix_initiaux[PRINCIPALE]), "après lissage": charges_choix(choix_lisses[PRINCIPALE])},
        f"{PRINCIPALE} : camions reçus par semaine et par jour (hors démarrage, camions réservés compris)")

Résultat (hors camions réservés) :

| Variante | Changements de profil | Surcoût du lissage | Coût après lissage | Semaines hors tolérance |
|---|---|---|---|---|
| 1 jour fixe | 93 | 1 117 € | 80 975 € | 20 → 2 |
| 1 ou 2 jours fixes | 125 | 1 517 € | 78 759 € | 20 → 1 |

Il reste une seule semaine hors tolérance (hors démarrage) : PLT_002 en S50, une petite semaine de 6 camions dont 4 le mardi : aucun changement de profil des couples de l'usine ne l'améliore plus.

### Bloc 4 : report et planning final

Rôle : réduire l'avance des livraisons sans ajouter de camion, puis construire le planning complet.

Report : un camion part le dernier jour fixe avant son premier besoin, mais il transporte souvent des lignes dont le besoin tombe le jour fixe suivant ou après. Ces lignes peuvent partir avec le camion du jour fixe suivant, s'il existe et s'il a la place. On n'ajoute jamais de camion. Ensuite, si un couple a plusieurs camions le même jour qui tiennent désormais dans un seul, on les regroupe. Le report n'est gardé pour un couple que s'il ne coûte rien, car le prix dépend du poids de chaque camion.

```
Couple au profil « mardi + vendredi »
camion du mardi 17/11 : besoins du 17/11, 18/11, 20/11, 23/11
  → les lignes du 20/11 et du 23/11 peuvent partir avec le camion du vendredi 20/11 (s'il a la place)
```

Planning final : on recalcule le contenu, le poids et le coût de chaque camion, puis on ajoute les camions réservés de la quarantaine. Le résultat est une ligne par camion (planning) et une ligne par ligne de besoin, avec son camion (contenu).

In [ ]:
def prochain_jour_fixe(date, profil):
    # Premier jour du profil strictement après `date`
    return min(date + pd.Timedelta(days=(j - date.day_of_week - 1) % 7 + 1) for j in profil)


def construire_planning(choix):
    # Camions d'un choix de profils (une ligne par camion, avec la liste de ses lignes)
    return pd.concat([scenarios[(p, s, prof)].assign(plant=p, supplier_id=s, profil=[prof] * len(scenarios[(p, s, prof)]))
                      for (p, s), prof in choix.items()], ignore_index=True)


def reporter_lignes(camions):
    # Report des lignes sur le jour fixe suivant + regroupement des camions d'un même jour, couple par couple,
    # gardé seulement s'il ne coûte rien
    besoin, poids, volume = besoins["need_date"], besoins["weight"], besoins["volume"]
    charge_ok = lambda ls: poids[ls].sum() <= POIDS_MAX + 1e-9 and volume[ls].sum() <= VOLUME_MAX + 1e-9
    resultat, journal = [], []
    for (plant, supplier), groupe in camions.groupby(["plant", "supplier_id"], sort=False):
        profil = groupe["profil"].iloc[0]
        cout = lambda ls: cout_camion(poids[ls].sum(), supplier)
        cam = [{"date": r.date, "demarrage": r.demarrage, "lignes": list(r.lignes)} for r in groupe.sort_values("date").itertuples()]
        cout_avant, nb_avant, reportees = sum(cout(c["lignes"]) for c in cam), len(cam), 0
        for c in cam:                                               # 1) report vers le jour fixe suivant
            suivant = prochain_jour_fixe(c["date"], profil)
            cibles = [u for u in cam if u["date"] == suivant and not u["demarrage"]]
            for l in sorted((l for l in c["lignes"] if besoin[l] >= suivant), key=lambda l: -poids[l]):
                for u in cibles:
                    if charge_ok(u["lignes"] + [l]):
                        c["lignes"].remove(l)
                        u["lignes"].append(l)
                        reportees += 1
                        break
        for date in sorted({c["date"] for c in cam}):                # 2) regroupement des camions d'un même jour
            du_jour = [c for c in cam if c["date"] == date and c["lignes"]]
            for a, b in itertools.combinations(du_jour, 2):
                if a["lignes"] and b["lignes"] and charge_ok(a["lignes"] + b["lignes"]):
                    a["lignes"] += b["lignes"]
                    b["lignes"] = []
        apres = [c for c in cam if c["lignes"]]
        cout_apres = sum(cout(c["lignes"]) for c in apres)
        garde = cout_apres <= cout_avant + 0.005
        if not garde:
            apres = [{"date": r.date, "demarrage": r.demarrage, "lignes": list(r.lignes)} for r in groupe.itertuples()]
        journal.append({"plant": plant, "supplier_id": supplier, "lignes_reportees": reportees if garde else 0,
                        "camions_en_moins": nb_avant - len(apres), "gain": round(cout_avant - cout_apres, 2) if garde else 0.0,
                        "report_refuse_car_plus_cher": not garde and reportees > 0})
        resultat += [{**c, "plant": plant, "supplier_id": supplier, "profil": profil} for c in apres]
    return pd.DataFrame(resultat), pd.DataFrame(journal)


def finaliser(camions):
    # Planning (une ligne par camion) et contenu (une ligne par ligne de besoin) à partir des camions
    camions = camions.sort_values(["plant", "date", "supplier_id"]).reset_index(drop=True)
    camions["camion_id"] = range(1, len(camions) + 1)
    affectation = camions[["camion_id", "date", "lignes"]].explode("lignes").set_index("lignes")
    lignes = besoins.join(affectation.rename(columns={"date": "date_livraison"}))
    lignes["avance_jours"] = (lignes["need_date"] - lignes["date_livraison"]).dt.days
    lignes.insert(0, "id_ligne", "L" + lignes.index.astype(str))
    contenu = lignes.groupby("camion_id").agg(poids=("weight", "sum"), volume=("volume", "sum"), nb_lignes=("weight", "size"))
    planning = camions.drop(columns=[c for c in ["lignes", "poids", "volume", "cout"] if c in camions]).join(contenu, on="camion_id")
    planning["cout"] = [cout_camion(p, s) for p, s in zip(planning["poids"], planning["supplier_id"])]
    planning["jours_fixes"] = planning["profil"].map(nom_profil)
    planning["reserve"] = False
    return planning, lignes


# ---------- Planning de chaque étape (pour le bilan) ----------
plannings = {}
for nom in VARIANTES:
    plannings[nom] = finaliser(construire_planning(choix_initiaux[nom]))
    plannings[f"{nom} + lissage"] = finaliser(construire_planning(choix_lisses[nom]))
camions_report, journal_report = reporter_lignes(construire_planning(choix_lisses[PRINCIPALE]))
plannings[f"{PRINCIPALE} + lissage + report"] = finaliser(camions_report)
planning, lignes_planning = plannings[f"{PRINCIPALE} + lissage + report"]
_, lignes_avant_report = plannings[f"{PRINCIPALE} + lissage"]

# ---------- Planning complet : camions optimisés + camions réservés (numérotés à la suite) ----------
decalage = int(planning["camion_id"].max())
reserves = camions_reserves.assign(camion_id=camions_reserves["numero"] + decalage, reserve=True, profil=None,
                                   jours_fixes="réservé : " + camions_reserves["motif"])
planning_complet = (pd.concat([planning, reserves.drop(columns=["numero", "motif"])], ignore_index=True)
                    .sort_values(["plant", "date", "supplier_id"]).reset_index(drop=True))
planning_complet["jour"] = planning_complet["date"].dt.day_of_week.map(dict(enumerate(JOURS)))
planning_complet["semaine"] = planning_complet["date"].dt.isocalendar().week
lignes_complet = pd.concat([lignes_planning.assign(reserve=False),
                            lignes_reserves.assign(camion_id=lignes_reserves["numero"] + decalage, reserve=True)
                            .drop(columns="numero")], ignore_index=True)

print(f"Report : {journal_report['lignes_reportees'].sum()} lignes reportées sur le jour fixe suivant "
      f"({journal_report['lignes_reportees'].gt(0).sum()} couples), {journal_report['camions_en_moins'].sum()} camion(s) en moins, "
      f"gain {eur(journal_report['gain'].sum())} ; refusé car plus cher pour {journal_report['report_refuse_car_plus_cher'].sum()} couples")
print(f"Avance moyenne d'une ligne : {nombre(lignes_avant_report['avance_jours'].mean())} j -> "
      f"{nombre(lignes_planning['avance_jours'].mean())} j ; lignes livrées le jour de leur besoin : "
      f"{(lignes_avant_report['avance_jours'] == 0).sum()} -> {(lignes_planning['avance_jours'] == 0).sum()}")
print(f"Planning complet : {len(planning_complet)} camions ({len(planning)} optimisés + {NB_RESERVE} réservés), "
      f"{eur(planning_complet['cout'].sum())}")
print("Couples par nombre de jours fixes :",
      planning.drop_duplicates(["plant", "supplier_id"])["profil"].map(len).value_counts().sort_index()
      .rename(lambda n: f"{n} jour(s)").to_dict())

In [ ]:
heatmap({"": charges_camions(planning_complet)},
        "Planning final : camions reçus par semaine et par jour (démarrage et camions réservés compris)")

In [ ]:
# Détail du planning d'une usine (une ligne par camion) : changer l'usine ici
USINE = "PLT_004"
(planning_complet.loc[planning_complet["plant"] == USINE,
                      ["camion_id", "supplier_id", "jours_fixes", "date", "jour", "semaine", "demarrage", "reserve",
                       "nb_lignes", "poids", "volume", "cout"]]
 .round({"poids": 1, "volume": 2}))

Résultat : le report fait partir 101 lignes (13 couples) avec le camion du jour fixe suivant, au lieu de 3 à 7 jours en avance. Les lignes livrées le jour même de leur besoin passent de 647 à 738. Le gain en coût est faible (70 €) et aucun camion n'est supprimé : un camion part toujours le dernier jour fixe avant son premier besoin, donc il ne peut jamais être reporté entier sans retard. Le report sert surtout à réduire le stock des usines.

Le planning final compte 736 camions (602 optimisés + 134 réservés) pour 112 195 €. 106 couples livrent sur 1 jour fixe, 67 sur 2 jours fixes.

### Choix du seuil de lissage


In [ ]:
# ---------- Trois indicateurs selon le seuil (de gauche à droite : on lisse de plus en plus) ----------
x = np.arange(len(etude))
graphes = [("Surcoût du lissage", "Ce que coûte le lissage (€)", "royalblue", eur),
           ("Jour le plus chargé (camions)", "Jour le plus chargé (camions)", "darkorange", lambda v: f"{v:.0f}"),
           ("Irrégularité moyenne", "Irrégularité moyenne des semaines", "seagreen", pourcent)]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), sharex=True)
for ax, (colonne, titre, couleur, fmt) in zip(axes, graphes):
    valeurs = etude[colonne] * (100 if colonne == "Irrégularité moyenne" else 1)
    ax.plot(x, valeurs, marker="o", color=couleur)
    for xi, (v, brut) in enumerate(zip(valeurs, etude[colonne])):
        if xi == 0 or brut != etude[colonne].iloc[xi - 1]:          # pas d'étiquette répétée
            ax.annotate(fmt(brut), (xi, v), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
    ax.axvline(ligne_retenue, color="red", linestyle="--", alpha=0.6)
    ax.text(ligne_retenue, ax.get_ylim()[1], " seuil retenu", color="red", va="top", fontsize=9)
    ax.set_title(titre)
    ax.set_xticks(x, etude["Seuil"], rotation=45, ha="right")
    ax.set_xlabel("Seuil (à droite : on lisse davantage)")
    ax.margins(y=0.15)
    ax.grid(alpha=0.3)
axes[2].yaxis.set_major_formatter(lambda v, _: f"{v:.0f} %")
fig.suptitle("Plus on baisse le seuil, plus on lisse : ce que ça coûte et ce que ça apporte", fontsize=13)
plt.tight_layout()
plt.show()


Comment lire les graphiques : de gauche à droite, on baisse le seuil, donc on lisse de plus en plus. Le coût monte (à gauche), tandis que le jour le plus chargé et l'irrégularité baissent (au milieu et à droite).

Ce qu'on en retient (camions réservés compris) :
- Le premier lissage fait l'essentiel : dès le seuil de 100 %, pour 1 065 €, le jour le plus chargé passe de 30 à 14 camions et l'irrégularité de 95 % à 48 %.
- Le jour le plus chargé ne descend jamais sous 14 camions : c'est la semaine S48 de PLT_002, la plus lourde (66 camions, environ 13 par jour, camions réservés compris). Ce n'est plus un pic, mais une semaine chargée partout.
- Jusqu'à 40 %, la régularité reste bon marché : de 100 % à 40 %, l'irrégularité passe de 48 % à 23 % pour 172 € de plus seulement.
- En dessous de 40 %, elle devient chère : passer à 35 % coûte 300 € pour 3,6 points d'irrégularité, soit environ 80 € par point, contre moins de 10 € par point entre 100 % et 40 %.

| Seuil | Coût total | Surcoût du lissage | Jour le plus chargé | Irrégularité moyenne |
|---|---|---|---|---|
| sans lissage | 110 659 € | 0 € | 30 camions | 95 % |
| 50 % | 111 846 € | 1 187 € | 14 camions | 27 % |
| 40 % | 111 895 € | 1 236 € | 14 camions | 23 % |
| 35 % (actuel) | 112 195 € | 1 536 € | 14 camions | 20 % |
| 15 % | 112 607 € | 1 949 € | 14 camions | 14 % |

Recommandation : avec les camions réservés, le meilleur compromis passe de 35 % à 40 %. Ça économise environ 300 € par rapport à 35 %, pour une irrégularité moyenne de 23 % au lieu de 20 %. Le notebook garde 35 % (`SEUIL_LISSAGE`) en attendant ta décision.


## 5. Validation des résultats

On vérifie le planning final avec les fonctions de la partie 1 (une ligne par contrainte), puis la cohérence du planning : chaque camion contient bien ce qu'il annonce, le poids livré est égal au poids demandé, et le coût total est bien la somme des coûts des camions.

In [ ]:
controle_contraintes = verifier_contraintes(planning_complet, lignes_complet, IDS_ATTENDUS)
controle_contraintes

In [ ]:
# Détail des cas particuliers : C9 (2 camions un même jour, inévitables) et C10 (semaines hors tolérance)
display(c9_un_camion_par_jour(planning_complet, lignes_complet))
c10_lissage(planning_complet, lignes_complet).style.format({"ecart": pourcent})

In [ ]:
def valider_resultats(planning, lignes):
    # Cohérence du planning final : contenu des camions, poids livré, coût
    opti, lignes_opti = optimises(planning), lignes[~lignes["reserve"]]
    contenu = lignes_opti.groupby("camion_id").agg(poids=("weight", "sum"), volume=("volume", "sum"), nb=("weight", "size"))
    p = opti.set_index("camion_id")
    controles = {
        "Chaque camion optimisé contient au moins une ligne": opti["camion_id"].isin(contenu.index).all(),
        "Poids de chaque camion = somme des poids de ses lignes": np.allclose(p["poids"], contenu.loc[p.index, "poids"]),
        "Volume de chaque camion = somme des volumes de ses lignes": np.allclose(p["volume"], contenu.loc[p.index, "volume"]),
        "Poids total livré = poids total des besoins": np.isclose(lignes_opti["weight"].sum(), besoins["weight"].sum()),
        "Un camion réservé par ligne sans poids": (lignes[lignes["reserve"]].groupby("camion_id").size() == 1).all()
                                                   and planning["reserve"].sum() == NB_RESERVE,
        "Coût de chaque camion = grille tarifaire": np.allclose(planning["cout"], [cout_camion(w, s) for w, s in zip(planning["poids"], planning["supplier_id"])]),
        "Coût total = somme des coûts des camions": np.isclose(planning["cout"].sum(), cout_total(planning)),
    }
    return pd.DataFrame({"Contrôle": list(controles), "Résultat": ["OK" if v else "ÉCHEC" for v in controles.values()]}).set_index("Contrôle")


validation = valider_resultats(planning_complet, lignes_complet)
assert (validation["Résultat"] == "OK").all(), "le planning final est incohérent"
assert (controle_contraintes.loc[controle_contraintes["Type"] == "dure", "Violations"] == 0).all(), "une contrainte dure est violée"
validation

Résultat : toutes les contraintes dures sont respectées et le planning est cohérent.
- C9 : 4 jours où un couple a 2 camions, tous inévitables. SUP_00037 → PLT_002 et SUP_00046 → PLT_004 ont un besoin de plus de 24,5 t un même jour (25,7 t et 46,7 t).
- C10 (souple) : 4 semaines hors tolérance sur 20.
  - Trois sont la semaine S46 de PLT_001, PLT_002 et PLT_004 : leur lundi 09/11 reçoit les camions de démarrage, qui ne peuvent être ni avancés (l'horizon commence ce jour-là) ni retardés.
  - La quatrième est la petite semaine S50 de PLT_002 : 6 camions, dont 4 le mardi.

## 6. Économies

Comparaison de chaque étape du moteur avec la baseline du milestone 2. Toutes les versions comprennent les mêmes 134 camions réservés (lignes de la quarantaine sans poids) : ils s'ajoutent au coût et à la charge des usines, sans changer l'économie en euros.

In [ ]:
def nb_hors_tolerance(camions):
    return sum(int(hors_tolerance(charge[CONTROLE]).sum()) for charge in charges_camions(camions).values())


versions = {"Baseline": (camions_base, None), "Algorithme de base (sans jours fixes)": finaliser(camions_algo)}
versions.update(plannings)
COUT_BASELINE_TOTAL = COUT_BASELINE + COUT_RESERVE
bilan = pd.DataFrame([{
    "": nom, "Camions": len(c) + NB_RESERVE, "Coût": c["cout"].sum() + COUT_RESERVE,
    "Économie vs baseline": COUT_BASELINE - c["cout"].sum(),
    "Économie (%)": (COUT_BASELINE - c["cout"].sum()) / COUT_BASELINE_TOTAL,
    "Avance moyenne (j)": np.nan if l is None else l["avance_jours"].mean(),
    "Couples à 2 jours": c.drop_duplicates(["plant", "supplier_id"])["profil"].map(len).eq(2).sum() if "jours_fixes" in c and nom in plannings else np.nan,
    f"Semaines hors tolérance (sur {len(USINES) * len(CONTROLE)})": nb_hors_tolerance(avec_reserves(c)),
} for nom, (c, l) in versions.items()]).set_index("")
bilan.style.format({"Coût": eur, "Économie vs baseline": eur, "Économie (%)": pourcent,
                    "Avance moyenne (j)": "{:.1f}", "Couples à 2 jours": "{:.0f}"}, na_rep="–")

In [ ]:
# Économies par usine (camions réservés compris)
base_u = avec_reserves(camions_base).groupby("plant").agg(camions_baseline=("cout", "size"), cout_baseline=("cout", "sum"))
final_u = planning_complet.groupby("plant").agg(camions_final=("cout", "size"), cout_final=("cout", "sum"))
par_usine = base_u.join(final_u)
par_usine.loc["Total"] = par_usine.sum()
par_usine["economie"] = par_usine["cout_baseline"] - par_usine["cout_final"]
par_usine["economie_pct"] = par_usine["economie"] / par_usine["cout_baseline"]
par_usine.style.format({"cout_baseline": eur, "cout_final": eur, "economie": eur, "economie_pct": pourcent,
                        "camions_baseline": "{:.0f}", "camions_final": "{:.0f}"})

In [ ]:
# Ce que coûtent les données manquantes : même calcul sans les lignes de la quarantaine (silver seul)
couples_quarantaine = set(map(tuple, besoins.loc[besoins.index >= N_SILVER, ["plant", "supplier_id"]].values))
FOND_VIDE = {plant: np.zeros((NB_SEMAINES, 5)) for plant in USINES}
choix_sans = {cle: prof for cle, prof in choix_initiaux[PRINCIPALE].items() if cle not in couples_quarantaine}
choix_sans, _ = lisser(choix_sans, VARIANTES[PRINCIPALE], fond=FOND_VIDE)
camions_sans, _ = reporter_lignes(construire_planning(choix_sans))
planning_sans, _ = finaliser(camions_sans)
planning_sans = planning_sans[planning_sans["camion_id"].notna()]
base_sans = camions_base[camions_base["source"] == "silver"]

qualite = pd.DataFrame({
    "Sans les lignes de la quarantaine": [base_sans["cout"].sum(), planning_sans["cout"].sum()],
    "Avec (camions réservés compris)": [COUT_BASELINE_TOTAL, planning_complet["cout"].sum()],
}, index=["Baseline", "Planning final"])
qualite.loc["Économie"] = qualite.loc["Baseline"] - qualite.loc["Planning final"]
qualite.loc["Économie (%)"] = qualite.loc["Économie"] / qualite.loc["Baseline"]
qualite.style.format(lambda v: pourcent(v) if abs(v) < 1 else eur(v))

Résultat :

| Version (camions réservés compris) | Camions | Coût | Économie vs baseline | Semaines hors tolérance |
|---|---|---|---|---|
| Baseline | 1 741 | 177 321 € | – | 12 / 20 |
| Algorithme de base (sans jours fixes) | 692 | 109 513 € | 67 807 € (-38,2 %) | 20 / 20 |
| 1 ou 2 jours fixes | 707 | 110 748 € | 66 572 € (-37,5 %) | 20 / 20 |
| 1 ou 2 jours fixes + lissage | 736 | 112 265 € | 65 055 € (-36,7 %) | 4 / 20 |
| Planning final (+ report) | 736 | 112 195 € | 65 126 € (-36,7 %) | 4 / 20 |

- Le moteur économise 65 126 € sur les 6 semaines du planning, soit -36,7 %, en divisant le nombre de camions par 2,4.
- Les jours fixes et le lissage ne coûtent que 2 682 € par rapport à l'algorithme de base sans contrainte de jours : on garde 96 % de l'économie maximale, avec des jours de livraison fixes et des usines équilibrées.
- Sans les lignes de la quarantaine, l'économie est de -45,8 %. L'écart vient des 134 camions réservés (33 506 €, 30 % du coût final) : sans le poids de ces lignes, on ne peut pas les consolider. Retrouver ces poids auprès du client est le plus gros levier d'économie restant.

## 7. Export

Les sorties du moteur sont écrites dans `data/gold/` :

| Fichier | Contenu | Une ligne par |
|---|---|---|
| `sortie1_profils.csv` | sortie 1 : jours fixes à appliquer à chaque couple, avec son nombre de camions et son coût | couple fournisseur × usine |
| `sortie2_camions.csv` | sortie 2 : le matériel regroupé dans chaque camion (ligne de besoin, camion, date de livraison, avance) | ligne de besoin |
| `planning_camions.csv` | planning complet : date, jours fixes, poids, volume et coût de chaque camion (camions réservés compris) | camion |

In [ ]:
sortie1 = (planning.groupby(["plant", "supplier_id", "jours_fixes"])
           .agg(nb_jours_fixes=("profil", lambda p: len(p.iloc[0])), camions=("camion_id", "size"),
                camions_demarrage=("demarrage", "sum"), poids_total=("poids", "sum"), cout=("cout", "sum"))
           .reset_index().round({"poids_total": 2, "cout": 2}))
sortie2 = (lignes_complet[["id_ligne", "camion_id", "plant", "supplier_id", "material", "unloading_point", "need_date",
                           "date_livraison", "avance_jours", "schedule_quantity", "weight", "volume", "reserve"]]
           .sort_values(["camion_id", "need_date"]))
planning_export = planning_complet[["camion_id", "plant", "supplier_id", "jours_fixes", "date", "jour", "semaine",
                                    "demarrage", "reserve", "nb_lignes", "poids", "volume", "cout"]]

assert sortie1[["plant", "supplier_id"]].duplicated().sum() == 0      # un profil par couple
assert len(sortie2) == len(IDS_ATTENDUS)                               # toutes les lignes
SORTIE.mkdir(parents=True, exist_ok=True)
sortie1.to_csv(SORTIE / "sortie1_profils.csv", index=False)
sortie2.to_csv(SORTIE / "sortie2_camions.csv", index=False)
planning_export.to_csv(SORTIE / "planning_camions.csv", index=False)
for nom, table in [("sortie1_profils.csv", sortie1), ("sortie2_camions.csv", sortie2), ("planning_camions.csv", planning_export)]:
    print(f"{nom:<22}: {len(table)} lignes")
sortie1.head()

## Conclusion

Le moteur produit le planning de livraison complet des 4 usines sur les 6 semaines, avec :
- des jours fixes pour chaque couple (106 couples sur 1 jour, 67 sur 2 jours) ;
- au plus 1 camion par couple et par jour, sauf 2 couples dont le besoin d'un seul jour dépasse 24,5 t ;
- des usines équilibrées : 4 semaines hors tolérance sur 20, dont 3 dues aux camions de démarrage du 09/11 ;
- toutes les contraintes dures respectées : capacité, aucune livraison en retard, au plus 7 jours d'avance, chaque ligne livrée une fois.

Il économise 65 126 € (-36,7 %) par rapport à la situation actuelle, et -45,8 % sur les lignes dont les données sont complètes.

Limites et pistes :
- Qualité des données : 134 lignes sans poids (30 % du coût final) ne peuvent pas être consolidées. Leur poids dépend du client.
- Ligne de plus de 24,5 t (SUP_00074 → PLT_003, 26,9 t le 15/12) : exclue en attendant la réponse du client.
- Seuil de lissage : 35 % est un compromis entre coût et régularité. Un seuil de 40 % coûterait environ 300 € de moins pour une régularité un peu moins bonne.
- Lissage glouton : il change un profil à la fois. Une recherche plus large (par exemple avec un solveur) pourrait encore réduire le surcoût.